# 01 — Data preparation

Produces the **frozen** splits in `data/processed/`; nothing downstream re-generates them.

1. TwitterEmo (`clarin-pl/twitteremo`) — 35,921 native Polish tweets
2. GoEmotions PL (NLLB-200) — 43,410 Reddit comments, EN→PL, labels remapped
3. CLARIN-Emo (`clarin-knext/CLARIN-Emo`) — Polish reviews, official PolEval splits

Pipeline: load → remap labels → clean + lemmatize (spaCy) → split 80/16/4 (`random_state=42`) → save.

## 1.1 Imports & Configuration

In [1]:
import os
import ast
import re
from pathlib import Path

import pandas as pd
import numpy as np
import spacy
from datasets import load_dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from dotenv import load_dotenv

load_dotenv()

# --- Stałe ---
RANDOM_STATE = 42
EMOTIONS = ["radość", "smutek", "zaufanie", "wstręt", "strach", "gniew", "przeczuwanie", "zdziwienie"]
# Sentiments (pozytywny, negatywny, neutralny, ambiwalentny, sarkazm) excluded from modeling:
# - TwitterEmo sentiments are largely derivable from emotions (redundant)
# - GoEmotions sentiments are synthetically derived in the mapping (not original annotations)
# - sarkazm is always 0 in GoEmotions, creating cross-dataset asymmetry
TARGET_LABELS = EMOTIONS

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

nlp = spacy.load("pl_core_news_lg")

print(f"spaCy model: {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Target labels ({len(TARGET_LABELS)}): {TARGET_LABELS}")

/path/to/repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


spaCy model: core_news_lg v3.8.0
Target labels (8): ['radość', 'smutek', 'zaufanie', 'wstręt', 'strach', 'gniew', 'przeczuwanie', 'zdziwienie']


## 1.2 Text preprocessing

Remove URLs, @mentions, `#`, standalone numbers → lemmatize (`pl_core_news_lg`) → drop stopwords and punctuation.

`pl_core_news_lg` over Morfeusz2: rule-based analyzers are linguistically better on standard Polish but fail on social-media OOV (slang, typos, abbreviations).

In [2]:
def clean_social_noise(text: str) -> str:
    """Strip social-media noise (URLs, @mentions, '#', standalone numbers) before lemmatization."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"\d+", "", text)
    return text


def lemmatize_doc(doc) -> str:
    """Join lowercased lemmas, dropping stopwords, punctuation and whitespace."""
    return " ".join(
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    )


def preprocess_dataset(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """Clean social-media noise, lemmatize with spaCy, and drop rows empty after cleaning.

    Lemmatization is batched with nlp.pipe for speed. Rows whose clean_text is empty
    after cleaning (no content-bearing tokens) are removed: they produce all-zero
    feature vectors for bag-of-words models and carry no usable signal.
    """
    df = df.copy()
    print(f"  Preprocessing {len(df)} rows (text column: '{text_col}')...")

    cleaned = df[text_col].apply(clean_social_noise).tolist()
    df["clean_text"] = [lemmatize_doc(doc) for doc in nlp.pipe(cleaned, batch_size=200)]

    empty_mask = df["clean_text"].str.strip() == ""
    empty_count = int(empty_mask.sum())
    print(f"  Empty after cleaning: {empty_count} ({empty_count / len(df) * 100:.1f}%) -> dropped")

    df = df[~empty_mask].reset_index(drop=True)
    print(f"  Remaining: {len(df)}")
    return df

## 1.3 Split helper

80/16/4, multi-label stratified, shared by all datasets.

In [3]:
def split_dataset(
    df: pd.DataFrame,
    label_cols: list[str] = TARGET_LABELS,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split into train/val/test = 80/16/4 with multi-label iterative stratification.

    Plain random splitting can leave rare labels (e.g. 'strach' ~0.9%) unevenly
    distributed across splits, destabilizing per-class F1 on the small test set.
    MultilabelStratifiedShuffleSplit balances the joint label distribution instead.
    """
    y = df[label_cols].to_numpy()

    # First split: 80% train / 20% temp
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=random_state)
    train_idx, temp_idx = next(msss.split(y, y))

    # Second split: temp -> 80% val / 20% test  (= 16% / 4% overall)
    y_temp = y[temp_idx]
    msss_temp = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=random_state)
    rel_val_idx, rel_test_idx = next(msss_temp.split(y_temp, y_temp))
    val_idx, test_idx = temp_idx[rel_val_idx], temp_idx[rel_test_idx]

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    print(f"  Split: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
    return train_df, val_df, test_df

---
## 2. TwitterEmo

`clarin-pl/twitteremo` — 35,921 tweets, 13 binary label columns (8 emotions + 5 sentiments).

In [4]:
# --- Load TwitterEmo ---
print("Loading TwitterEmo from HuggingFace...")
dataset_tw = load_dataset("clarin-pl/twitteremo", token=os.environ.get("HF_TOKEN"))
df_tw = dataset_tw["train"].to_pandas()

print(f"Loaded: {len(df_tw)} rows, columns: {list(df_tw.columns)}")
print(f"\nLabel columns present: {[c for c in TARGET_LABELS if c in df_tw.columns]}")
print(f"Text column: 'tekst'")
df_tw.head(3)

Loading TwitterEmo from HuggingFace...
Loaded: 35921 rows, columns: ['id', 'data', 'tekst', 'radość', 'smutek', 'zaufanie', 'wstręt', 'strach', 'gniew', 'przeczuwanie', 'zdziwienie', 'pozytywny', 'negatywny', 'neutralny', 'ambiwalentny', 'sarkazm']

Label columns present: ['radość', 'smutek', 'zaufanie', 'wstręt', 'strach', 'gniew', 'przeczuwanie', 'zdziwienie']
Text column: 'tekst'


,id,data,tekst,radość,smutek,zaufanie,wstręt,strach,gniew,przeczuwanie,zdziwienie,pozytywny,negatywny,neutralny,ambiwalentny,sarkazm
0,0,2021-09-02,@anonymized_account jeszcze zaraz będzie rady ...,0,0,0,0,0,0,0,0,0,0,1,0,0
1,1,2021-09-02,Komisja Europejska szantażuje polski rząd? Pol...,0,0,0,0,0,1,0,0,0,1,0,0,0
2,2,2021-09-02,@anonymized_account Ten kto go zaprosił życzy ...,0,0,0,1,0,1,0,0,0,1,0,0,1


In [5]:
# --- Preprocess TwitterEmo ---
print("TwitterEmo preprocessing:")
df_tw = preprocess_dataset(df_tw, text_col="tekst")

# --- Deduplicate on raw text before splitting ---
# Duplicate tweets exist in the raw dataset; splitting before dedup causes the same
# text to appear in multiple splits (verified in section 4 leakage check).
n_before = len(df_tw)
df_tw = df_tw.drop_duplicates(subset="tekst", keep="first").reset_index(drop=True)
n_removed = n_before - len(df_tw)
print(f"\n  Duplicates removed: {n_removed} ({n_removed / n_before * 100:.1f}%)")
print(f"  Remaining: {len(df_tw)}")

# --- Split ---
print("\nTwitterEmo split:")
tw_train, tw_val, tw_test = split_dataset(df_tw)

# --- Verify label distribution ---
print("\nLabel distribution (train):")
for label in TARGET_LABELS:
    count = tw_train[label].sum()
    print(f"  {label:>15}: {count:>5} ({count / len(tw_train) * 100:>5.1f}%)")

TwitterEmo preprocessing:
  Preprocessing 35921 rows (text column: 'tekst')...
  Empty after cleaning: 33 (0.1%) -> dropped
  Remaining: 35888

  Duplicates removed: 32 (0.1%)
  Remaining: 35856

TwitterEmo split:
  Split: train=28684, val=5737, test=1435

Label distribution (train):
           radość:  3290 ( 11.5%)
           smutek:  1335 (  4.7%)
         zaufanie:  1289 (  4.5%)
           wstręt:  6655 ( 23.2%)
           strach:   258 (  0.9%)
            gniew:  5065 ( 17.7%)
     przeczuwanie: 10069 ( 35.1%)
       zdziwienie:  1868 (  6.5%)


In [6]:
# --- Save TwitterEmo splits ---
# 'sarkazm' is kept as an extra column (NOT part of TARGET_LABELS): it has native
# annotations only in TwitterEmo (always 0 in GoEmotions, absent in CLARIN-Emo), so it
# cannot be modeled cross-dataset. Preserved here for an optional TwitterEmo-only study.
tw_cols = ["tekst", "clean_text"] + TARGET_LABELS + ["sarkazm"]

tw_train[tw_cols].to_csv(PROCESSED_DIR / "twitteremo_train.csv", index=False)
tw_val[tw_cols].to_csv(PROCESSED_DIR / "twitteremo_val.csv", index=False)
tw_test[tw_cols].to_csv(PROCESSED_DIR / "twitteremo_test.csv", index=False)

print("Saved TwitterEmo splits to data/processed/")
print(f"  train: {len(tw_train)}, val: {len(tw_val)}, test: {len(tw_test)}")
print(f"  sarkazm prevalence: train={tw_train['sarkazm'].mean() * 100:.1f}%, "
      f"val={tw_val['sarkazm'].mean() * 100:.1f}%, test={tw_test['sarkazm'].mean() * 100:.1f}%")

Saved TwitterEmo splits to data/processed/
  train: 28684, val: 5737, test: 1435
  sarkazm prevalence: train=2.0%, val=2.1%, test=3.1%


---
## 3. GoEmotions PL (translated)

English Reddit comments translated with NLLB-200-distilled-600M; 43,410 rows. Original 28 labels
remapped to the 8 Plutchik emotions. **Known limitation:** translation quality is imperfect for
informal text — quantified in EDA section 11.

### Label mapping: GoEmotions (28) → Plutchik (8)

| TwitterEmo | GoEmotions source labels | Notes |
|---|---|---|
| radość | joy, amusement, excitement, pride, relief, love, admiration, gratitude | `love` debatable (trust component) |
| smutek | sadness, disappointment, grief, remorse, embarrassment | `embarrassment` debatable |
| zaufanie | approval, caring | |
| wstręt | disgust | |
| strach | fear, nervousness | |
| gniew | anger, annoyance, disapproval | intensity collapse (mild→strong) |
| przeczuwanie | optimism, desire | |
| zdziwienie | surprise, confusion, realization, curiosity | `confusion` debatable |
| *(brak emocji)* | neutral | texts with `neutral` label get all-zero emotion rows |

Sentiments (pozytywny, negatywny, neutralny, ambiwalentny, sarkazm) are **excluded from modeling** — see configuration cell for rationale.

In [7]:
GO_EMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]

# Mapping GoEmotions 28 labels → Plutchik's 8 basic emotions (TwitterEmo schema).
# "neutral" is intentionally excluded: texts with no active emotion have all-zero rows.
# Debatable mappings (noted for thesis discussion):
#   love → radość  (has trust component, could argue zaufanie)
#   annoyance/disapproval → gniew  (intensity collapse: mild → strong anger)
#   confusion → zdziwienie  (distinct cognitive state mapped to surprise)
#   embarrassment → smutek  (self-conscious emotion, not prototypical sadness)
LABEL_MAPPING = {
    "radość":        ["joy", "amusement", "excitement", "pride", "relief", "love", "admiration", "gratitude"],
    "smutek":        ["sadness", "disappointment", "grief", "remorse", "embarrassment"],
    "zaufanie":      ["approval", "caring"],
    "wstręt":        ["disgust"],
    "strach":        ["fear", "nervousness"],
    "gniew":         ["anger", "annoyance", "disapproval"],
    "przeczuwanie":  ["optimism", "desire"],
    "zdziwienie":    ["surprise", "confusion", "realization", "curiosity"],
}


def map_goemotions_to_twitteremo(df: pd.DataFrame) -> pd.DataFrame:
    """Remap GoEmotions 28-label schema to 8 Plutchik emotions (TwitterEmo schema)."""
    df = df.copy()

    df["labels"] = df["labels"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    label_names = df["labels"].apply(lambda ids: [GO_EMOTIONS_LABELS[i] for i in ids])

    for target_emo, source_list in LABEL_MAPPING.items():
        source_set = set(source_list)
        df[target_emo] = label_names.apply(
            lambda names: int(bool(source_set & set(names)))
        )

    return df

In [8]:
# --- Load GoEmotions (translated CSV, with repetition-loop fixes) ---
# go_emotions_pl_nllb_fixed.csv = NLLB output where 44 repetition-loop translations were
# re-translated with anti-loop decoding (see fixing_wrong_translations.ipynb).
# The original go_emotions_pl_nllb_final.csv is preserved untouched for provenance.
print("Loading GoEmotions PL from CSV...")
df_go_raw = pd.read_csv(RAW_DIR / "go_emotions_pl_nllb_fixed.csv")
print(f"Loaded: {len(df_go_raw)} rows")

# --- Remap labels ---
print("\nRemapping GoEmotions labels to TwitterEmo schema...")
df_go = map_goemotions_to_twitteremo(df_go_raw)

print(f"\nLabel distribution:")
for label in TARGET_LABELS:
    count = df_go[label].sum()
    print(f"  {label:>15}: {count:>5} ({count / len(df_go) * 100:>5.1f}%)")

print(f"\nRows with no emotion label: {(df_go[EMOTIONS].sum(axis=1) == 0).sum()}")
df_go[["text", "text_pl"] + TARGET_LABELS].head(5)

Loading GoEmotions PL from CSV...
Loaded: 43410 rows

Remapping GoEmotions labels to TwitterEmo schema...

Label distribution:
           radość: 12442 ( 28.7%)
           smutek:  3263 (  7.5%)
         zaufanie:  3961 (  9.1%)
           wstręt:   793 (  1.8%)
           strach:   726 (  1.7%)
            gniew:  5579 ( 12.9%)
     przeczuwanie:  2158 (  5.0%)
       zdziwienie:  5367 ( 12.4%)

Rows with no emotion label: 12823


,text,text_pl,radość,smutek,zaufanie,wstręt,strach,gniew,przeczuwanie,zdziwienie
0,My favourite food is anything I didn't have to...,"Moim ulubionym jedzeniem jest wszystko, czego ...",0,0,0,0,0,0,0,0
1,"Now if he does off himself, everyone will thin...","Jeśli się zabije, wszyscy pomyślą, że śmieje s...",0,0,0,0,0,0,0,0
2,WHY THE FUCK IS BAYLESS ISOING,Dlaczego kurwa jest bezbaylsowa?,0,0,0,0,0,1,0,0
3,To make her feel threatened,Aby poczuć się zagrożona,0,0,0,0,1,0,0,0
4,Dirty Southern Wankers,Śmierdzący południowi Wankers,0,0,0,0,0,1,0,0


In [9]:
# --- Preprocess GoEmotions PL ---
print("GoEmotions PL preprocessing:")
df_go = preprocess_dataset(df_go, text_col="text_pl")

# --- Deduplicate on translated text before splitting (mirrors TwitterEmo) ---
# Reddit comments contain frequent duplicates ("Yep of course.", "Thank you", ...);
# splitting before dedup leaks identical model inputs across train/val/test.
n_before = len(df_go)
df_go = df_go.drop_duplicates(subset="text_pl", keep="first").reset_index(drop=True)
n_removed = n_before - len(df_go)
print(f"\n  Duplicates removed: {n_removed} ({n_removed / n_before * 100:.1f}%)")
print(f"  Remaining: {len(df_go)}")

# --- Split ---
print("\nGoEmotions PL split:")
go_train, go_val, go_test = split_dataset(df_go)

# --- Verify label distribution (train) ---
print("\nLabel distribution (train):")
for label in TARGET_LABELS:
    count = go_train[label].sum()
    print(f"  {label:>15}: {count:>5} ({count / len(go_train) * 100:>5.1f}%)")

GoEmotions PL preprocessing:
  Preprocessing 43410 rows (text column: 'text_pl')...
  Empty after cleaning: 199 (0.5%) -> dropped
  Remaining: 43211

  Duplicates removed: 366 (0.8%)
  Remaining: 42845

GoEmotions PL split:
  Split: train=34276, val=6855, test=1714

Label distribution (train):
           radość:  9770 ( 28.5%)
           smutek:  2598 (  7.6%)
         zaufanie:  3135 (  9.1%)
           wstręt:   630 (  1.8%)
           strach:   574 (  1.7%)
            gniew:  4430 ( 12.9%)
     przeczuwanie:  1718 (  5.0%)
       zdziwienie:  4244 ( 12.4%)


In [10]:
# --- Save GoEmotions splits ---
# Keep both original English text and Polish translation for cross-lingual analysis
go_cols = ["text", "text_pl", "clean_text"] + TARGET_LABELS

go_train[go_cols].to_csv(PROCESSED_DIR / "go_emotions_train.csv", index=False)
go_val[go_cols].to_csv(PROCESSED_DIR / "go_emotions_val.csv", index=False)
go_test[go_cols].to_csv(PROCESSED_DIR / "go_emotions_test.csv", index=False)

print("Saved GoEmotions PL splits to data/processed/")
print(f"  train: {len(go_train)}, val: {len(go_val)}, test: {len(go_test)}")

Saved GoEmotions PL splits to data/processed/
  train: 34276, val: 6855, test: 1714


---
## 3.5 CLARIN-Emo — cross-domain evaluation set

Polish consumer reviews (PolEval 2024 Task 2), 6 annotators, label kept when ≥2/6 agree. Same
Plutchik scheme as TwitterEmo, different domain. Official PolEval splits preserved (no re-split).
Sentiment columns dropped; `sarkazm` does not exist here.

| CLARIN-Emo | TwitterEmo | CLARIN-Emo | TwitterEmo |
|---|---|---|---|
| Joy | radość | Anger | gniew |
| Sadness | smutek | Anticipation | przeczuwanie |
| Trust | zaufanie | Surprise | zdziwienie |
| Disgust | wstręt | Fear | strach |

In [11]:
# --- Load CLARIN-Emo (PolEval 2024 Task 2) — cross-domain evaluation set ---
print("Loading CLARIN-Emo from HuggingFace...")
clarin_emo = load_dataset("clarin-knext/CLARIN-Emo", token=os.environ.get("HF_TOKEN"))

# CLARIN-Emo emotion column -> TwitterEmo label (1:1 on Plutchik's 8)
CLARIN_EMO_MAP = {
    "Joy": "radość",
    "Sadness": "smutek",
    "Trust": "zaufanie",
    "Disgust": "wstręt",
    "Fear": "strach",
    "Anger": "gniew",
    "Anticipation": "przeczuwanie",
    "Surprise": "zdziwienie",
}


def prepare_clarin_emo(split_df: pd.DataFrame) -> pd.DataFrame:
    """Remap CLARIN-Emo to the TwitterEmo schema: raw 'tekst' + 8 binary emotion columns.

    Sentiment columns are dropped (only emotions are modeled). Boolean labels are cast
    to int to match the 0/1 encoding used by the other datasets.
    """
    df = split_df.rename(columns={"text": "tekst", **CLARIN_EMO_MAP})
    df[TARGET_LABELS] = df[TARGET_LABELS].astype(int)
    return df[["tekst"] + TARGET_LABELS]


# Official PolEval splits are preserved (no re-split) for benchmark comparability.
clarin_cols = ["tekst", "clean_text"] + TARGET_LABELS
for split in ("train", "val", "test"):
    df_ce = prepare_clarin_emo(clarin_emo[split].to_pandas())
    print(f"\nCLARIN-Emo [{split}] preprocessing:")
    df_ce = preprocess_dataset(df_ce, text_col="tekst")
    df_ce[clarin_cols].to_csv(PROCESSED_DIR / f"clarin_emo_{split}.csv", index=False)
    print(f"  Saved clarin_emo_{split}.csv ({len(df_ce)} rows)")

# --- Label distribution (test split = primary cross-domain eval set) ---
ce_test = pd.read_csv(PROCESSED_DIR / "clarin_emo_test.csv")
print("\nLabel distribution (clarin_emo test):")
for label in TARGET_LABELS:
    count = int(ce_test[label].sum())
    print(f"  {label:>15}: {count:>5} ({count / len(ce_test) * 100:>5.1f}%)")

Loading CLARIN-Emo from HuggingFace...

CLARIN-Emo [train] preprocessing:
  Preprocessing 7169 rows (text column: 'tekst')...
  Empty after cleaning: 802 (11.2%) -> dropped
  Remaining: 6367
  Saved clarin_emo_train.csv (6367 rows)

CLARIN-Emo [val] preprocessing:
  Preprocessing 1401 rows (text column: 'tekst')...
  Empty after cleaning: 168 (12.0%) -> dropped
  Remaining: 1233
  Saved clarin_emo_val.csv (1233 rows)

CLARIN-Emo [test] preprocessing:
  Preprocessing 1431 rows (text column: 'tekst')...
  Empty after cleaning: 175 (12.2%) -> dropped
  Remaining: 1256
  Saved clarin_emo_test.csv (1256 rows)

Label distribution (clarin_emo test):
           radość:   550 ( 43.8%)
           smutek:   550 ( 43.8%)
         zaufanie:   209 ( 16.6%)
           wstręt:   243 ( 19.3%)
           strach:    59 (  4.7%)
            gniew:   210 ( 16.7%)
     przeczuwanie:   125 ( 10.0%)
       zdziwienie:    92 (  7.3%)


---
## 4. Verification

Quick sanity checks on saved files.

In [12]:
# --- Verify saved files ---
print("=== Saved files ===\n")
for f in sorted(PROCESSED_DIR.glob("*.csv")):
    df_check = pd.read_csv(f)
    print(f"{f.name:>30}: {len(df_check):>6} rows, {len(df_check.columns):>2} cols")


def check_leakage(name: str, text_col: str, prefix: str) -> None:
    """Assert train/val/test are disjoint on the given text column."""
    print(f"\n=== Data leakage check ({name}) ===")
    train = set(pd.read_csv(PROCESSED_DIR / f"{prefix}_train.csv")[text_col])
    val = set(pd.read_csv(PROCESSED_DIR / f"{prefix}_val.csv")[text_col])
    test = set(pd.read_csv(PROCESSED_DIR / f"{prefix}_test.csv")[text_col])

    overlap_tv, overlap_tt, overlap_vt = train & val, train & test, val & test
    print(f"  train ∩ val:  {len(overlap_tv)} overlapping texts")
    print(f"  train ∩ test: {len(overlap_tt)} overlapping texts")
    print(f"  val ∩ test:   {len(overlap_vt)} overlapping texts")

    if len(overlap_tv) + len(overlap_tt) + len(overlap_vt) == 0:
        print("  ✓ No data leakage detected")
    else:
        print("  ✗ WARNING: Data leakage found!")


check_leakage("TwitterEmo", text_col="tekst", prefix="twitteremo")
check_leakage("GoEmotions", text_col="text_pl", prefix="go_emotions")

=== Saved files ===

           clarin_emo_test.csv:   1256 rows, 10 cols
          clarin_emo_train.csv:   6367 rows, 10 cols
            clarin_emo_val.csv:   1233 rows, 10 cols
          go_emotions_test.csv:   1714 rows, 11 cols
         go_emotions_train.csv:  34276 rows, 11 cols
           go_emotions_val.csv:   6855 rows, 11 cols
           twitteremo_test.csv:   1435 rows, 11 cols
          twitteremo_train.csv:  28684 rows, 11 cols
            twitteremo_val.csv:   5737 rows, 11 cols

=== Data leakage check (TwitterEmo) ===
  train ∩ val:  0 overlapping texts
  train ∩ test: 0 overlapping texts
  val ∩ test:   0 overlapping texts
  ✓ No data leakage detected

=== Data leakage check (GoEmotions) ===
  train ∩ val:  0 overlapping texts
  train ∩ test: 0 overlapping texts
  val ∩ test:   0 overlapping texts
  ✓ No data leakage detected


In [13]:
# --- Sample: translation quality preview ---
print("=== GoEmotions: Translation quality samples ===\n")
go_sample = go_train.sample(10, random_state=RANDOM_STATE)
for _, row in go_sample.iterrows():
    active_labels = [l for l in TARGET_LABELS if row[l] == 1]
    print(f"EN: {row['text']}")
    print(f"PL: {row['text_pl']}")
    print(f"    Labels: {active_labels}")
    print("---")

=== GoEmotions: Translation quality samples ===

EN: Great deal for us, happy he's back. Up next, [NAME]! 
PL: Świetnie dla nas, szczęśliwy, że wrócił.
    Labels: ['radość']
---
EN: Maybe next time... Hold that 40 pt L for now.
PL: Może następnym razem... zatrzymaj 40 pt L na razie.
    Labels: []
---
EN: Nah man, name the kid the entire Bee Movie script
PL: No stary, nazwij dziecko całym scenariuszem Bee Movie
    Labels: []
---
EN: Thanks friend lol
PL: Dzięki przyjacielu lol
    Labels: ['radość']
---
EN: someone should people have been commenting on the old tweet the past little while, as it got resurfaced
PL: Ktoś powinien był komentować stary tweet od krótkiego czasu, gdy pojawił się na nowo
    Labels: []
---
EN: He has no side of the story or we would have heard it already.
PL: Nie ma żadnej strony historii, bo inaczej już byśmy ją słyszeli.
    Labels: []
---
EN: I am an atheist and the idea of a god makes me disgusted and angry so I prefer not to be religious
PL: Jestem atei

---
## Summary

Split sizes are computed from the saved files below, so they never go stale. All three datasets
use the same 8 Plutchik labels.

**Excluded from modeling:** sentiment columns — redundant w.r.t. emotions in TwitterEmo,
synthetically derived in GoEmotions. **Exception:** `sarkazm` is kept as an extra column in the
TwitterEmo CSVs only (native annotations, ~2% prevalence); not in `TARGET_LABELS`.